# Capstone: debug a teammate's notebook

Your teammate wrote this notebook to answer three questions about the green taxi trips of October to December 2017:
1. Which pickup zones are the busiest?
2. For long trips (over 10 miles), which pickup zones get the highest average tip?
3. How many trips were store-and-forward trips (flag `Y`)? `Y` means the taxi's meter had no connection,
   so it stored the trip and sent it later.

The notebook contains **exactly four bugs**. Your job: find them, fix them, and record them.

**How to work**
1. Pick **Serverless** in the compute menu at the top of the notebook. You need to have run `00-setup-data.ipynb` once.
2. **Run all from the top**: click **Run all** at the top of the notebook.
3. Databricks stops at the first cell that raises an error.
   Read the error the Session 4 way. First read the last line: the error's name and its message.
   Then find the line that names your cell (`File "<cell 2>"...`;
   on Databricks it should read `File <command-...>`).
4. Fill in one row of the findings table. Then fix the cause. The mistake may sit in a different cell from the error.
5. Click **Run all** again. Repeat until every cell runs to the end.
6. Then check the answers, because **one bug raises no error at all**.
   Facts to compare with: the data has 2,705,926 trips, and a few thousand of them are store-and-forward trips.

Each code cell starts with a comment such as `# Cell 4`. Use those numbers in the table.

## Findings table

Double-click this cell to edit it.

| Bug | Cell where it showed up | Cell with the mistake | Error name or symptom | Kind of bug | Your fix |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |

The kinds of bug from the course: **run order**, **imports**, **a column error that surfaces at an action**,
and **a schema type error** (a silent null).

## The teammate's notebook

In [ ]:
# Cell 1: setup
BASE = "/Volumes/workspace/default/bdcc"
from pyspark.sql import functions as F

In [ ]:
# Cell 2: check the columns and their types
trips.printSchema()

In [ ]:
# Cell 3: read the trips with a schema (faster than letting Spark guess the types)
schema = """
VendorID INT,
lpep_pickup_datetime TIMESTAMP,
lpep_dropoff_datetime TIMESTAMP,
store_and_fwd_flag INT,
RatecodeID INT,
PULocationID INT,
DOLocationID INT,
passenger_count INT,
trip_distance FLOAT,
fare_amount FLOAT,
extra FLOAT,
mta_tax FLOAT,
tip_amount FLOAT,
tolls_amount FLOAT,
ehail_fee FLOAT,
improvement_surcharge FLOAT,
total_amount FLOAT,
payment_type INT,
trip_type INT
"""
trips = spark.read.csv(
    f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv",
    sep=",",
    header=True,
    schema=schema,
)

In [ ]:
# Cell 4: look at a few rows
trips.show(5)

In [ ]:
# Cell 5: question 1, the busiest pickup zones
busiest = trips.groupBy("PULocationID").count().orderBy(desc("count"), "PULocationID")
busiest.show(5)

In [ ]:
# Cell 6: keep the long trips (over 10 miles)
long_trips = trips.filter(F.col("trip_distance") > 10)

In [ ]:
# Cell 7: average tip per pickup zone, for long trips
zone_tips = long_trips.groupBy("PULocationID").agg(F.mean("tip_amont").alias("mean_tip"))

In [ ]:
# Cell 8: sort the zones, highest average tip first
top_tips = zone_tips.orderBy(F.desc("mean_tip"), "PULocationID")

In [ ]:
# Cell 9: question 2, the zones with the highest average tip on long trips
top_tips.show(5)

In [ ]:
# Cell 10: question 3, how many trips were store-and-forward ("Y")?
trips.groupBy("store_and_fwd_flag").count().orderBy("store_and_fwd_flag").show()

## Your answers

When every cell runs and all four bugs are fixed, write the answers here:
1. The busiest pickup zone and its number of trips:
2. The pickup zone with the highest average tip on long trips:
3. The number of store-and-forward trips: